# Lab09 — Clean up: leave nothing running

**Two situations, two paths. Pick one.**

| Your situation | Path | What happens |
| --- | --- | --- |
| **A. Lab00 created a project for this workshop** and nothing else lives in it | §9.2 **Shut the project down** | billing unlinked, project deleted after 30 days, every resource with it |
| **B. You ran the workshop in a project you keep** (your own project, shared with other work) | §9.3 **Remove the workshop resources, keep the project** | every resource the labs created is deleted one by one; APIs, IAM you had before and the project stay |

Both paths are dry runs until you flip their confirmation switch, and both print every command so you can run them yourself.

**You will learn**
1. What shutting down a project does, and why billing is unlinked first
2. Which resources the nine labs created, and how to remove exactly those

Estimated time: 5 minutes.
Measured run time (fresh project, September 2026): 1 min for path A, 14 min for path B (about sixty delete calls, run one after another); reading adds to it.

> **Terminal or notebook — your choice.** Every cell that calls a CLI prints the exact command first (`$ …`).
> Copy it into your own terminal (from the repo root) if you prefer to run it yourself; the cells just automate the same commands.


In [ ]:
# --- Workshop configuration (same cell at the top of every lab) ---
import os, sys, json, pathlib
from dotenv import load_dotenv

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
ENV_FILE = REPO_ROOT / "workshop.env"
assert ENV_FILE.exists(), "workshop.env not found - run Lab00 first"
load_dotenv(ENV_FILE, override=True)

PROJECT_ID      = os.environ["PROJECT_ID"]
PROJECT_NUMBER  = os.environ["PROJECT_NUMBER"]
REGION          = os.environ["REGION"]           # europe-west1: Agent Runtime, Sessions, Memory Bank, Gateway, Model Armor
MODEL_LOCATION  = os.environ["MODEL_LOCATION"]   # eu: multi-region endpoint that serves gemini-3.8-flash
MODEL           = os.environ["MODEL"]            # gemini-3.8-flash
AGENT_NAME      = os.environ["AGENT_NAME"]       # nova-assistant
AGENT_DIR       = REPO_ROOT / AGENT_NAME

# Make every shell (!) and SDK call in this notebook target the workshop project.
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "GOOGLE_GENAI_USE_VERTEXAI": "true",
    "CLOUDSDK_CORE_PROJECT": PROJECT_ID,
    "CLOUDSDK_CORE_DISABLE_PROMPTS": "1",
})
print(f"Project: {PROJECT_ID} ({PROJECT_NUMBER}) | region: {REGION} | model: {MODEL} @ {MODEL_LOCATION}")
print(f"Agent dir: {AGENT_DIR}")

def terminal(cmd, cwd=None):
    """Print the exact command a cell is about to run, so you can copy/paste it into your own terminal."""
    prefix = f"cd {os.path.relpath(cwd, REPO_ROOT)} && " if cwd else ""
    print(f"$ {prefix}{cmd}\n")

## 9.1 What shutting down a project does

The Resource Manager documentation states it directly: *"Shutting down a project stops all billing and resource usage."*
The timeline:

| When | What happens |
| --- | --- |
| immediately | the project is marked for deletion and becomes unusable; *"Any associated Cloud Billing account is disconnected"* |
| 30 days | recovery window (`lifecycleState: DELETE_REQUESTED`); an Owner can restore it with `gcloud projects undelete` |
| after 30 days | *"the project and all its resources are permanently deleted and can't be recovered"* |

Two details worth knowing before you press the button:

1. **Unlink billing first.** The same page notes that *"Projects may continue to incur charges until the current billing
   cycle ends"* and recommends disabling billing before the shutdown. The cell below does exactly that: unlink, then delete.
2. **The project id is gone for good.** Ids are never reused, and the project counts against your project quota until the
   30 days are over. Lab00 puts the date into the id so the next run gets a fresh one.

Nothing the workshop created lives outside the project: the organization-policy override from Lab06, the IAM access
policies and the registry entries are all project-scoped. **Only take this path for a project that exists just for the workshop.** What stays on your machine costs nothing: the `geap-workshop`
gcloud configuration, `workshop.env`, the `nova-assistant/` folder and the coding-agent skills from Lab00.

Docs: [Shut down (delete) and restore projects](https://docs.cloud.google.com/resource-manager/docs/delete-restore-projects).


## 9.2 Path A — shut down the workshop project

For a project that Lab00 created for this workshop. If you want to try the optional [Lab08](lab08_gemini_enterprise.ipynb)
(Gemini Enterprise), do it first. The cell is a dry run until you set `CONFIRM_DELETE = True`.


In [ ]:
# --- Shut down the project: unlink billing -> delete the project -> confirm the state -> switch gcloud back ---
import subprocess

# Safety catch: nothing below runs until you set this to True.
CONFIRM_DELETE = False

steps = [
    f"gcloud billing projects unlink {PROJECT_ID}",                              # stop charges now, not at the end of the billing cycle
    f"gcloud projects delete {PROJECT_ID} --quiet",                              # marks the project for deletion (30-day recovery window)
    f"gcloud projects describe {PROJECT_ID} --format='value(lifecycleState)'",  # expect DELETE_REQUESTED
    "gcloud config configurations activate default",                            # back to your usual gcloud setup
]
for cmd in steps:
    terminal(cmd)
    if CONFIRM_DELETE:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print((r.stdout or r.stderr).strip(), "\n")

if not CONFIRM_DELETE:
    print("dry run - set CONFIRM_DELETE = True and re-run this cell to execute the four commands")
else:
    print(f"project {PROJECT_ID} is shutting down; restore within 30 days with: gcloud projects undelete {PROJECT_ID}")


## 9.3 Path B — keep the project, remove every workshop resource

For a project you keep. The cell below deletes, in dependency order, everything the labs created, and only that:

| Lab | Resources removed |
| --- | --- |
| 07 | online monitors, the evaluation results bucket `gs://<PROJECT_ID>-eval` |
| 02–06 | the Agent Runtime instance `nova-assistant` with its sessions, memories and sandboxes |
| 06 | gateway authorization policies and extensions, the Agent Gateway `nova-egress-gateway`, the IAM access policy and its binding, the project-level org-policy override |
| 03, 06 | Cloud Run services `nova-inventory-mcp` and `nova-returns-agent`, their source bucket and the `cloud-run-source-deploy` image repository |
| 03, 06 | Agent Registry entries: the warehouse MCP server, the returns desk, the `gapi-*` endpoints; the private skill `nova-sales-analytics` |
| 05 | Model Armor templates `nova-guard-input` and `nova-guard-output`, the log-based metric `nova_blocked_prompts` |
| 03 | BigQuery dataset `nova_shop` |
| 08 | the Gemini Enterprise app `nova-workplace` and its data store `nova-kb` (only if Lab08 created them) |
| 03, 05, 06 | project IAM bindings of the agent's identity and of the gateway service agent |

What stays, on purpose: the APIs enabled in Lab00, the roles Lab03 and Lab06 granted to the default compute service account (other
workloads may rely on them), the Observability Analytics upgrade of the `_Default` log bucket (one-way, no cost), and logs and traces
until their retention expires. Every step is best-effort: a resource that does not exist is reported and skipped.

The cell is a dry run until you set `CONFIRM_RESOURCE_CLEANUP = True`.


In [ ]:
# --- Path B: delete every workshop resource in dependency order, keep the project ---
import subprocess, json

# Safety catch: nothing below runs until you set this to True.
CONFIRM_RESOURCE_CLEANUP = False

# Names and ids recorded by the labs in workshop.env (missing ones simply skip their step).
env = os.environ
AGENT_URL   = env.get("NOVA_AGENT_URL", "")                       # Lab02: https://REGION-aiplatform.googleapis.com/v1/projects/NUM/locations/REGION/reasoningEngines/ID
PRINCIPAL   = env.get("NOVA_AGENT_PRINCIPAL", "")                 # Lab03: principal://agents.global.org-.../reasoningEngines/ID
SKILLS_LOC  = env.get("NOVA_SKILLS_LOCATION", "eu")               # Lab03
GE_APP      = env.get("GE_APP_NAME", "")                          # Lab08: projects/NUM/locations/eu/collections/default_collection/engines/nova-workplace
GATEWAY_SA  = subprocess.run(f"gcloud network-services agent-gateways describe nova-egress-gateway --location={REGION} --project={PROJECT_ID} --format='value(agentGatewayCard.serviceExtensionsServiceAccount)' 2>/dev/null",
                             shell=True, capture_output=True, text=True).stdout.strip()   # read before the gateway is deleted below
API_EW1     = f"https://{REGION}-aiplatform.googleapis.com/v1beta1/projects/{PROJECT_ID}/locations/{REGION}"
TOKEN       = 'H="Authorization: Bearer $(gcloud auth print-access-token)"'

def run(label, cmd):
    """Echo one clean-up command; execute it only when confirmed; report done / not found / failed without stopping."""
    terminal(cmd)
    if not CONFIRM_RESOURCE_CLEANUP:
        return
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode == 0:
        print(f"  done: {label}")
    elif "NOT_FOUND" in out or "not found" in out.lower() or "does not exist" in out.lower():
        print(f"  not found (already gone): {label}")
    else:
        print(f"  FAILED: {label}\n    {out[-300:]}")

def names(cmd):
    """Return the lines a list command prints (empty when the resource type has no entries)."""
    return [l for l in subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.split() if l]

# 1. Lab07: online monitors, then the evaluation results bucket.
for oe in names(f'curl -s {API_EW1}/onlineEvaluators -H "Authorization: Bearer $(gcloud auth print-access-token)" | python3 -c "import json,sys; [print(o[\'name\']) for o in json.load(sys.stdin).get(\'onlineEvaluators\', [])]"'):
    run(f"online monitor {oe.split('/')[-1]}", f'curl -s -X DELETE https://{REGION}-aiplatform.googleapis.com/v1beta1/{oe} -H {TOKEN.split("=",1)[1]}')
run("evaluation results bucket", f"gcloud storage rm -r gs://{PROJECT_ID}-eval --project={PROJECT_ID}")

# 2. Lab02-06: the Agent Runtime instance; force=true removes its sessions, memories and sandboxes as well.
if AGENT_URL:
    run("Agent Runtime instance nova-assistant (with sessions, memories, sandboxes)", f'curl -s -X DELETE "{AGENT_URL}?force=true" -H "Authorization: Bearer $(gcloud auth print-access-token)"')

# 3. Lab06: gateway policies and extensions, the gateway, the IAM access policy and binding, the org-policy override.
for pol in ["nova-iap-authz-policy", "nova-ma-content-policy"]:
    run(f"authorization policy {pol}", f"gcloud network-security authz-policies delete {pol} --location={REGION} --project={PROJECT_ID} --quiet")
for ext in ["nova-iap-authz-ext", "nova-ma-content-ext"]:
    run(f"authorization extension {ext}", f"gcloud beta service-extensions authz-extensions delete {ext} --location={REGION} --project={PROJECT_ID} --quiet")
run("Agent Gateway nova-egress-gateway", f"gcloud network-services agent-gateways delete nova-egress-gateway --location={REGION} --project={PROJECT_ID} --quiet")
run("IAM access policy binding", f"gcloud iam policy-bindings delete nova-assistant-egress-binding --location=global --project={PROJECT_ID} --quiet")
run("IAM access policy nova-assistant-egress", f"gcloud iam access-policies delete nova-assistant-egress --location=global --project={PROJECT_ID} --quiet")
run("org-policy override iam.managed.disableAccessPolicyBinding", f"gcloud org-policies delete iam.managed.disableAccessPolicyBinding --project={PROJECT_ID}")

# 4. Lab03, Lab06: the two mock systems on Cloud Run, their source bucket and image repository.
for svc in ["nova-inventory-mcp", "nova-returns-agent"]:
    run(f"Cloud Run service {svc}", f"gcloud run services delete {svc} --region={REGION} --project={PROJECT_ID} --quiet")
run("Cloud Run source bucket", f"gcloud storage rm -r gs://run-sources-{PROJECT_ID}-{REGION} --project={PROJECT_ID}")
run("Artifact Registry repository cloud-run-source-deploy", f"gcloud artifacts repositories delete cloud-run-source-deploy --location={REGION} --project={PROJECT_ID} --quiet")

# 5. Lab03, Lab06: Agent Registry entries (warehouse MCP, returns desk, every gapi-* endpoint) and the private skill.
for entry in names(f"gcloud agent-registry services list --project={PROJECT_ID} --location={REGION} --format='value(name.basename())'"):
    if entry.startswith(("nova-", "gapi-")):
        run(f"registry entry {entry}", f"gcloud agent-registry services delete {entry} --project={PROJECT_ID} --location={REGION} --quiet")
for rev in names(f"gcloud alpha agent-registry skills revisions list --skill=private-nova-sales-analytics --location={SKILLS_LOC} --project={PROJECT_ID} --format='value(name.basename())' 2>/dev/null"):
    run(f"skill revision {rev}", f"gcloud alpha agent-registry skills revisions delete {rev} --skill=private-nova-sales-analytics --location={SKILLS_LOC} --project={PROJECT_ID} --quiet")
run("private skill nova-sales-analytics", f"gcloud alpha agent-registry skills delete private-nova-sales-analytics --location={SKILLS_LOC} --project={PROJECT_ID} --quiet")

# 6. Lab05: Model Armor templates and the log-based metric.
for tpl in ["nova-guard-input", "nova-guard-output"]:   # templates are regional: address the regional endpoint, as Lab05 did
    run(f"Model Armor template {tpl}", f'curl -s -f -X DELETE https://modelarmor.{REGION}.rep.googleapis.com/v1/projects/{PROJECT_ID}/locations/{REGION}/templates/{tpl} -H "Authorization: Bearer $(gcloud auth print-access-token)"')
run("log-based metric nova_blocked_prompts", f"gcloud logging metrics delete nova_blocked_prompts --project={PROJECT_ID} --quiet")

# 7. Lab03: the sample dataset.
run("BigQuery dataset nova_shop", f"bq rm -r -f -d {PROJECT_ID}:nova_shop")

# 8. Lab08 (only if it ran): the Gemini Enterprise app, then its data store.
if GE_APP:
    ge_loc = GE_APP.split("/")[3]
    run("Gemini Enterprise app nova-workplace", f'curl -s -X DELETE https://{ge_loc}-discoveryengine.googleapis.com/v1/{GE_APP} -H "Authorization: Bearer $(gcloud auth print-access-token)" -H "X-Goog-User-Project: {PROJECT_ID}"')
    run("Gemini Enterprise data store nova-kb", f'curl -s -X DELETE https://{ge_loc}-discoveryengine.googleapis.com/v1/projects/{PROJECT_ID}/locations/{ge_loc}/collections/default_collection/dataStores/nova-kb -H "Authorization: Bearer $(gcloud auth print-access-token)" -H "X-Goog-User-Project: {PROJECT_ID}"')

# 9. Lab03, Lab05, Lab06: project IAM bindings the labs granted to the agent's identity and to the gateway service agent.
if PRINCIPAL:
    for role in ["roles/bigquery.dataViewer", "roles/bigquery.jobUser", "roles/mcp.toolUser", "roles/serviceusage.serviceUsageConsumer", "roles/agentregistry.viewer", "roles/modelarmor.user"]:
        run(f"IAM {role} for the agent identity", f'gcloud projects remove-iam-policy-binding {PROJECT_ID} --member="{PRINCIPAL}" --role={role} --condition=None --quiet >/dev/null')
for role in (["roles/modelarmor.calloutUser", "roles/serviceusage.serviceUsageConsumer", "roles/modelarmor.user"] if GATEWAY_SA else []):
    run(f"IAM {role} for the gateway service agent", f"gcloud projects remove-iam-policy-binding {PROJECT_ID} --member=serviceAccount:{GATEWAY_SA} --role={role} --condition=None --quiet >/dev/null")

if not CONFIRM_RESOURCE_CLEANUP:
    print("dry run - set CONFIRM_RESOURCE_CLEANUP = True and re-run this cell to execute the commands above")


In [ ]:
# --- Path B check: what is left of the workshop in the project ---
checks = {
    "Agent Runtime instances":  f"gcloud beta ai reasoning-engines list --region={REGION} --project={PROJECT_ID} --format='value(displayName)' 2>/dev/null || curl -s https://{REGION}-aiplatform.googleapis.com/v1/projects/{PROJECT_ID}/locations/{REGION}/reasoningEngines -H \"Authorization: Bearer $(gcloud auth print-access-token)\" | python3 -c \"import json,sys; [print(r.get('displayName')) for r in json.load(sys.stdin).get('reasoningEngines', [])]\"",
    "Cloud Run services":       f"gcloud run services list --region={REGION} --project={PROJECT_ID} --format='value(metadata.name)'",
    "Agent Gateways":           f"gcloud network-services agent-gateways list --location={REGION} --project={PROJECT_ID} --format='value(name)'",
    "registry entries nova-/gapi-": f"gcloud agent-registry services list --project={PROJECT_ID} --location={REGION} --format='value(name.basename())' | grep -E '^(nova-|gapi-)' || true",
    "Model Armor templates":    f"curl -s https://modelarmor.{REGION}.rep.googleapis.com/v1/projects/{PROJECT_ID}/locations/{REGION}/templates -H \"Authorization: Bearer $(gcloud auth print-access-token)\" | python3 -c \"import json,sys; [print(t['name'].split('/')[-1]) for t in json.load(sys.stdin).get('templates', [])]\"",
    "BigQuery dataset":         f"bq ls --project_id={PROJECT_ID} --format=prettyjson 2>/dev/null | python3 -c \"import json,sys; [print(d['datasetReference']['datasetId']) for d in json.load(sys.stdin) if d['datasetReference']['datasetId']=='nova_shop']\"",
    "buckets":                  f"gcloud storage buckets list --project={PROJECT_ID} --format='value(name)'",
}
for label, cmd in checks.items():
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    print(f"{label:<30} {out.replace(chr(10), ', ') if out else '- none -'}")


## Recap — the whole journey

| Lab | Nova Assistant gained | Platform capability |
| --- | --- | --- |
| 01 | tools, instruction, local testing | ADK, Agents CLI, playground |
| 02 | a home in the cloud, persistent sessions, discoverability | Agent Runtime, Agent Registry |
| 03 | real data, a warehouse connection, domain skills, an analyst | BigQuery remote MCP server, custom MCP server on Cloud Run, Agent Registry, Skills |
| 04 | memory, safe computation, feedback, transparency | Sessions, Memory Bank, Code Execution, Feedback service, metrics, logs, traces |
| 05 | protection against injection and leaks | Model Armor (two templates, plugin, floor settings), alerts |
| 06 | governed access to enterprise systems, Model Armor on the network path | Agent Identity, Agent Gateway, IAM access policies, Model Armor `CONTENT_AUTHZ` |
| 07 | proof of quality | evaluation datasets, judges, user simulation, compare |
| 08 *(optional)* | a place in the employee-facing app | Gemini Enterprise (ADK and A2A registration) |
| 09 | a clean exit | project shutdown, billing unlink, resource clean-up |
